# Initial Steps:

1. Access the attached files in the classroom and the data link on the Kaggle website.
2. Download the complete dataset, read its contents, and analyze the relationships between the data points.
3. Download Docker and configure its settings.
4. Open Docker and enter SQL commands to create the tables and assign the PK and FK values ​​according to the table relationships.
5. Verify that the table has been created correctly.
6. Connected to PostgreSQL from a Jupyter Notebook using Python and SQLAlchemy.
7. Compared CSV row counts with database row counts to validate the ingestion process.
8. Execute some join commands to ensure that the FK values ​​are correctly selected.
all screen shots are attached.

In [17]:
# load datasets

import pandas as pd

df = pd.read_csv("../data/raw/olist_orders_dataset.csv")

df.head()
df.shape
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [ ]:
# Convert Date Columns to Datetime
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


In [ ]:
# Check Uniqueness of order_id.
df["order_id"].duplicated().sum()
df["order_id"].is_unique


In [ ]:
# Load environment variables from the .env file
 
import os
from dotenv import load_dotenv

load_dotenv()

print(os.getenv("DB_HOST"))
print(os.getenv("DB_PORT"))
print(os.getenv("DB_USER"))
print(os.getenv("DB_NAME"))

localhost
5432
postgres
learning_db


In [ ]:
# Establish DB connection
from sqlalchemy import create_engine

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)
with engine.connect() as connection:
    print("Database connection successful!")

Database connection successful!


In [ ]:
# SELECT query 
import pandas as pd
query = "SELECT COUNT(*) FROM orders;"

result = pd.read_sql(query, engine)

result

,count
0,99441


In [ ]:
# Validate Database Data
import pandas as pd
orders_csv = pd.read_csv("../data/raw/olist_orders_dataset.csv")

print("CSV rows:", len(orders_csv))

db_orders = pd.read_sql(
    "SELECT COUNT(*) AS count FROM orders;",
    engine
)

print("Database rows:", db_orders["count"][0])

CSV rows: 99441
Database rows: 99441


In [18]:
# Ensure that the database matches the original dataset
tables = {
    "customers": "../data/raw/olist_customers_dataset.csv",
    "geolocation": "../data/raw/olist_geolocation_dataset.csv",
    "orders": "../data/raw/olist_orders_dataset.csv",
    "order_items": "../data/raw/olist_order_items_dataset.csv",
    "order_payments": "../data/raw/olist_order_payments_dataset.csv",
    "order_reviews": "../data/raw/olist_order_reviews_dataset.csv",
    "products": "../data/raw/olist_products_dataset.csv",
    "sellers": "../data/raw/olist_sellers_dataset.csv",
    "product_category_name_translation": "../data/raw/product_category_name_translation.csv"
}

for table, csv_path in tables.items():
    csv_count = len(pd.read_csv(csv_path))
    db_count = pd.read_sql(
        f"SELECT COUNT(*) AS count FROM {table};",
        engine
    )["count"][0]

    print(f"{table}: CSV = {csv_count}, DB = {db_count}")

customers: CSV = 99441, DB = 99441
geolocation: CSV = 1000163, DB = 1000163
orders: CSV = 99441, DB = 99441
order_items: CSV = 112650, DB = 112650
order_payments: CSV = 103886, DB = 103886
order_reviews: CSV = 99224, DB = 99224
products: CSV = 32951, DB = 32951
sellers: CSV = 3095, DB = 3095
product_category_name_translation: CSV = 71, DB = 71


In [15]:
# JOIN query to ensure the relationship between orders and customers.

query = """
SELECT
    o.order_id,
    o.customer_id,
    o.order_status,
    c.customer_city,
    c.customer_state
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
LIMIT 10;
"""

result = pd.read_sql(query, engine)

result

,order_id,customer_id,order_status,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,santo andre,SP
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,congonhinhas,PR
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,santa rosa,RS
7,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,nilopolis,RJ
8,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,faxinalzinho,RS
9,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,sorocaba,SP


In [ ]:
#Another JOIN query to ensure the relationship between `orders` and `order_items`.
query = """
SELECT
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp,
    oi.product_id,
    oi.seller_id,
    oi.price
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
LIMIT 10;
"""

result = pd.read_sql(query, engine)

result

,order_id,order_status,order_purchase_timestamp,product_id,seller_id,price
0,00010242fe8c5a6d1ba2dd792cb16214,delivered,2017-09-13 08:59:02,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,58.90
1,00018f77f2f0320c557190d7a144bdd3,delivered,2017-04-26 10:53:06,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,239.90
2,000229ec398224ef6ca0657da4fc703e,delivered,2018-01-14 14:33:31,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,199.00
3,00024acbcdf0a6daa1e931b038114c75,delivered,2018-08-08 10:00:35,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,12.99
4,00042b26cf59d7ce69dfabb4e55b4fd9,delivered,2017-02-04 13:57:51,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,199.90
5,00048cc3ae777c65dbb7d2a0634bc1ea,delivered,2017-05-15 21:42:34,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,21.90
6,00054e8431b9d7675808bcb819fb4a32,delivered,2017-12-10 11:53:48,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,19.90
7,000576fe39319847cbb9d288c5617fa6,delivered,2018-07-04 12:08:27,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,810.00
8,0005a1a1728c9d785b8e2b08b904576c,delivered,2018-03-19 18:40:33,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,145.95
9,0005f50442cb953dcd1d21e1fb923495,delivered,2018-07-02 13:59:39,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,53.99


# Relationships
- customers 1──►N orders (customer_id)
- orders 1──►N order_reviews (order_id)
- orders 1──►N order_payments (order_id)
- orders 1──►N order_items (order_id)
- sellers 1──►N order_items (seller_id)
- products 1──►N order_items (product_id)

# Result
I modified the notebook based on the mentor's instructions. Initially, I did everything directly using Docker. Now, I have modified the workflow to use Python and Jupyter Notebook.

- The dataset is running locally in PostgreSQL using Docker.
- Successfully connected to the database using Python.
- Loaded the raw dataset downloaded from Kaggle and compared it with the data stored in the local database to ensure that the database matches the original dataset.
- Executed SQL SELECT and JOIN queries to test and verify the data and relationships between the tables.